In [ ]:
# 04 — Train the LSTM (the actual model). Loads the windows from 02, trains the
# LSTM for ~60 epochs keeping the weights with the best validation MAE, scores
# it on the test set against the 03 baselines, and packages the result as a
# versioned Weights & Biases artifact (weights + scaler + config) that the
# serving pipeline loads.

In [8]:
# 04 — Train the LSTM (the actual model). Loads the windows from 02, trains the
# LSTM keeping the best-validation weights, scores it against the 03 baselines,
# and logs a versioned W&B artifact that the serving pipeline loads.

import sys; from pathlib import Path
ROOT = Path.cwd(); ROOT = ROOT if (ROOT/"common").exists() else ROOT.parent  # works from either folder
sys.path.insert(0, str(ROOT/"modeling"))
import numpy as np, pandas as pd, torch, wandb
from torch.utils.data import DataLoader
from dataset import SequenceDataset
from model import LSTMRegressor

# Fix the random seeds.
torch.manual_seed(0); np.random.seed(0)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# Load the pre-built, pre-scaled windows saved by notebook 02.
z = np.load(ROOT/"modeling"/"artifacts"/"windows.npz")
# Wrap each split in a DataLoader, which feeds the model data in batches.
# Training data is shuffled; validation isn't (order doesn't matter for scoring).
train_dl = DataLoader(SequenceDataset(z["Xtr"], z["ytr"]), batch_size=64, shuffle=True)
val_dl   = DataLoader(SequenceDataset(z["Xva"], z["yva"]), batch_size=256)
n_features = z["Xtr"].shape[-1]   # 13 — the number of features per hour
print("device:", device, "| features:", n_features, "| train batches:", len(train_dl))

device: mps | features: 13 | train batches: 700


In [9]:
config = dict(hidden=64, layers=1, dropout=0.0, lr=1e-3, epochs=60, batch=64, seq_len=48, horizon=24)
wandb.init(project="airquality-pm25", config=config)

model = LSTMRegressor(n_features, config["hidden"], config["layers"], config["dropout"]).to(device)
opt = torch.optim.Adam(model.parameters(), lr=config["lr"])
loss_fn = torch.nn.MSELoss()

def mae_on(dl):                                          # MAE in raw µg/m³ (y is unscaled)
    model.eval(); errs = []
    with torch.no_grad():
        for xb, yb in dl:
            errs.append((model(xb.to(device)).cpu() - yb).abs())
    return float(torch.cat(errs).mean())

In [10]:
# Train for many passes over the data, keeping the best version seen.
best_mae, best_state = float("inf"), None
for epoch in range(config["epochs"]):
    model.train()   # learning mode
    # One pass over the training data, one batch at a time.
    for xb, yb in train_dl:
        opt.zero_grad()                                    # clear last step's gradients
        loss = loss_fn(model(xb.to(device)), yb.to(device))  # how wrong is this batch?
        loss.backward()                                    # compute how to adjust the weights
        opt.step()                                         # apply that adjustment
    # After each epoch, score on the validation set.
    val_mae = mae_on(val_dl)
    wandb.log({"epoch": epoch, "train_loss": float(loss), "val_mae": val_mae})
    # Keep a copy of the weights whenever validation error hits a new low.
    if val_mae < best_mae:
        best_mae = val_mae
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
print("best val MAE:", round(best_mae, 3))

best val MAE: 4.418


In [11]:
# Load the best weights we saved, then define a helper to predict on raw arrays.
model.load_state_dict(best_state)
def preds_on(X):
    model.eval()
    with torch.no_grad():
        return model(torch.from_numpy(X).float().to(device)).cpu().numpy().ravel()

# Score the final model on the TEST set
yte, pte = z["yte"], preds_on(z["Xte"])
lstm_mae  = float(np.abs(pte - yte).mean())
lstm_rmse = float(np.sqrt(((pte - yte) ** 2).mean()))

# Put the LSTM's scores next to the baselines
bl = pd.read_csv(ROOT/"modeling"/"artifacts"/"baseline_metrics.csv")
board = pd.concat([bl, pd.DataFrame([{"model": "lstm", "MAE": round(lstm_mae,3), "RMSE": round(lstm_rmse,3)}])])
print(board.to_string(index=False))
wandb.log({"test_mae": lstm_mae, "test_rmse": lstm_rmse})

# Save the trained weights to disk.
torch.save(best_state, ROOT/"modeling"/"artifacts"/"model.pt")

# Also save the model's shape (how many features/units/layers) so the serving
# pipeline can rebuild the exact same network from the artifact alone.
import json
model_cfg = {"n_features": n_features, "hidden": config["hidden"],
             "layers": config["layers"], "dropout": config["dropout"]}
(ROOT/"modeling"/"artifacts"/"model_config.json").write_text(json.dumps(model_cfg))

# Bundle the three files the serving pipeline needs into one versioned W&B
# artifact: the weights, the scaler + windowing config, and the architecture.
# Inference pins a specific version of this so retraining can't silently change
# what production predicts.
art = wandb.Artifact("pm25-lstm", type="model")
art.add_file(str(ROOT/"modeling"/"artifacts"/"model.pt"))
art.add_file(str(ROOT/"modeling"/"artifacts"/"preprocess.joblib"))
art.add_file(str(ROOT/"modeling"/"artifacts"/"model_config.json"))
wandb.log_artifact(art); wandb.finish()
print("saved model.pt + model_config.json + logged W&B artifact")

               model   MAE  RMSE
seasonal_persistence 2.879 5.019
         climatology 4.094 5.294
             xgboost 2.847 4.276
                lstm 2.790 4.138


epoch,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇███
test_mae,▁
test_rmse,▁
train_loss,▄▁▂▁▁▁▂▁▁█▁▂▂▁▁▁▁▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_mae,▂▁▂▂▃▅▄▄▅▄▄▅▅▆▇▆▇▇▇▆▇▇▇▇▇▇█▇▇█▇▇▇▇█▇██▇█
epoch,59
test_mae,2.78988
test_rmse,4.13834
train_loss,9.96708
val_mae,6.50159


saved model.pt + model_config.json + logged W&B artifact
